In [1]:
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.dynamic_factor import DynamicFactor
from numpy.testing import assert_allclose
import os

# Assuming you have these from your repo's test data

from statsmodels.tsa.statespace.tests.results import results_dynamic_factor, results_varmax

current_path = os.getcwd()  # current working directory (your notebook location)
output_path = os.path.join("results", "results_dynamic_factor_stata.csv")

# Full path to the CSV
csv_path = os.path.join(current_path, output_path)

# Load the CSV
output_results = pd.read_csv(csv_path)

print(output_results.head())


   predict_dfm_1  predict_dfm_2  predict_dfm_3  predict_dfm2_1  \
0            NaN            NaN            NaN             NaN   
1       0.000000       0.000000       0.000000        0.000000   
2       0.016978       0.017588       0.016944        0.003488   
3       0.024179       0.025048       0.024131        0.006008   
4       0.027629       0.028622       0.027574        0.004347   

   predict_dfm2_2  predict_dfm2_3  predict_dfm_exog1_1  predict_dfm_exog1_2  \
0             NaN             NaN                  NaN                  NaN   
1        0.000000        0.000000             0.018033             0.020667   
2        0.003545        0.003529             0.018142             0.020731   
3        0.006106        0.006078             0.023122             0.023647   
4        0.004419        0.004399             0.020469             0.022094   

   predict_dfm_exog1_3  predict_dfm_exog2_1  ...  dyn_predict_dfm_scalar_2  \
0                  NaN                  NaN  ...  

In [38]:
# Copy reference results
true = results_dynamic_factor.lutkepohl_dfm_exog1.copy()

# Add prediction and dynamic prediction columns from CSV
true["predict"] = output_results.iloc[76:][[
    "predict_dfm_exog1_1",
    "predict_dfm_exog1_2",
    "predict_dfm_exog1_3"
]]
true["dynamic_predict"] = output_results.iloc[76:][[
    "dyn_predict_dfm_exog1_1",
    "dyn_predict_dfm_exog1_2",
    "dyn_predict_dfm_exog1_3"
]]


In [39]:
exog_train = np.ones((75, 1))

In [40]:
dta = pd.DataFrame(
            results_varmax.lutkepohl_data, columns=['inv', 'inc', 'consump'],
            index=pd.date_range('1960-01-01', '1982-10-01', freq='QS'))

dta['inv'] = np.log(dta['inv']).diff()
dta['inc'] = np.log(dta['inc']).diff()
dta['consump'] = np.log(dta['consump']).diff()

endog = dta.loc['1960-04-01':'1978-10-01']

# if demean:
#     endog -= dta.iloc[1:][included_vars].mean()



In [41]:
print(endog)

                 inv       inc   consump
1960-04-01 -0.005571  0.030570  0.014354
1960-07-01  0.032970  0.042111  0.030412
1960-10-01  0.037140  0.016360  0.031749
1961-01-01  0.094363  0.031939  0.024257
1961-04-01 -0.043590  0.021381 -0.002181
...              ...       ...       ...
1977-10-01  0.026188  0.021032  0.017272
1978-01-01  0.025520  0.010843  0.012479
1978-04-01  0.035580  0.014599  0.018431
1978-07-01  0.025508  0.024339  0.013194
1978-10-01  0.036368  0.005173  0.005990

[75 rows x 3 columns]


In [42]:
mod = DynamicFactor(endog, k_factors=1, factor_order=1, exog=exog_train)
res = mod.fit(disp=False)


/opt/anaconda3/envs/pymc_extras/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [44]:
exog_pred = np.ones((16, 1))

# Standard predictions
pred = res.get_prediction(start=1, end=16, exog=exog_pred).predicted_mean
print(pred.shape)
print(true["predict"].shape)
assert_allclose(pred, true["predict"], atol=1e-5)



(16, 3)
(16, 3)


AssertionError: 
Not equal to tolerance rtol=1e-07, atol=1e-05

Mismatched elements: 47 / 48 (97.9%)
Max absolute difference: 0.00441398
Max relative difference: 0.32929094
 x: array([[0.017818, 0.02056 , 0.019703],
       [0.020094, 0.021719, 0.021012],
       [0.019265, 0.021297, 0.020536],...
 y: array([[0.013404, 0.017957, 0.017354],
       [0.016592, 0.019823, 0.01906 ],
       [0.017584, 0.020405, 0.019591],...

In [ ]:
# Dynamic predictions
dyn_pred = res.get_prediction(start=1, end=16, dynamic=True, exog=exog_pred).predicted_mean
assert_allclose(dyn_pred, true["dynamic_predict"], atol=1e-5)

In [ ]:
bse = res._cov_params_approx().diagonal()**0.5
assert_allclose(bse**2, true['var_oim'], atol=1e-5)